<a href="https://colab.research.google.com/github/nainikadevireddy/JohnsHopkinsAI/blob/main/Deep%20Neural%20Networks/6%3A%20Bacteria%20Competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<small><font color=gray>Notebook author: <a href="https://www.linkedin.com/in/olegmelnikov/" target="_blank">Oleg Melnikov</a> ©2021 onwards</font></small><hr style="margin:0;background-color:silver">

**<font size=6>🦠Bacteria</font>**. [**Instructions**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=ITaPDPIQEgXV) for running Colabs.

<details>
  <summary><small>Sharing consent: <mark>[ X ]</mark></summary>
  <div>
We consent to sharing our Colab (after the assignment ends) with other students/instructors for educational purposes. We understand that sharing is <b>optional</b> and this decision will not affect our grade in any way. <font color=gray><i>
Instructions: If ok with sharing your Colab for educational purposes, leave "X" in the check box.</i></font></small></div>

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')   # OK to enable, if your kaggle.json is stored in Google Drive

In [ ]:
!pip install inflect==7.0.0 >> log  # resolves pip's dependency issue for inflect 7.4.0 requires typeguard>=4.0.1
!pip install -U tensorflow_addons >> log
!pip install 'keras<3.0.0' mediapipe-model-maker >>log  # fix from https://github.com/google-ai-edge/mediapipe/issues/5229

In [ ]:
#!pip install --upgrade --force-reinstall --no-deps kaggle >> log  # upgrade kaggle package (to avoid a warning)
!mkdir -p ~/.kaggle                               # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >>log  # First, download kaggle.json from kaggle.com (in Account page) and place it in the root of mounted Google Drive
!cp kaggle.json ~/.kaggle/kaggle.json >> log       # Alternative location of kaggle.json (without a connection to Google Drive)
!chmod 600 ~/.kaggle/kaggle.json                  # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v 24feb25-bacteria # set the competition context for the next few kaggle API calls. !kaggle config view - shows current settings
!kaggle competitions download >> log              # download competition dataset as a zip file
!unzip -o *.zip >> log                            # Kaggle dataset is copied as a single file and needs to be unzipped.
!kaggle competitions leaderboard --show           # print public leaderboard

cp: cannot stat '/content/drive/MyDrive/kaggle.json': No such file or directory
- competition is now set to: 24feb25-bacteria
Using competition: 24feb25-bacteria
  teamId  teamName                   submissionDate       score         
--------  -------------------------  -------------------  ------------  
13430225  Team_1_David_Josh          2025-03-09 03:56:45  0.9895110526  
13434827  Ray Tu                     2025-03-09 00:44:44  0.9882814466  
13432872  12_SichaoLiu_ShrinitBabel  2025-03-09 17:29:30  0.9881909248  
13421931  Team_14_Seth_Shuai         2025-03-09 16:00:55  0.9880867380  
13452211  13_Henderson_Willis        2025-03-09 17:01:10  0.9878650593  
13443743  7_Caz_Jungwoo              2025-03-09 06:21:15  0.9871372252  
13434110  4 Mai Simmons              2025-03-09 01:44:36  0.9862066692  
13459728  Team 3                     2025-03-08 17:01:29  0.9860332204  
13434582  9_Nainika_Madihah          2025-03-08 21:53:17  0.9857169448  
13432194  10_Yang_Daniel           

In [ ]:
%%time
%%capture
%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, seaborn as sns, os, tensorflow as tf, tensorflow.keras as keras
from keras.layers import Flatten, Dense
from keras.models import Sequential
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
os.environ['TF_DETERMINISTIC_OPS'] = '1'; os.environ['TF_CUDNN_DETERMINISTIC'] = '1'; # allows seeding RNG on GPU
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=100, precision=2, edgeitems=2, suppress=True)
pd.set_option('display.max_columns', 20, 'display.precision', 2, 'display.max_rows', 4)

CPU times: user 4.68 s, sys: 785 ms, total: 5.46 s
Wall time: 7.03 s


In [ ]:
df = pd.read_csv('XY_Bacteria.csv'); df

,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0,y
0,10,32,41,39,77,122,55,81,58,31,...,38,88,80,20,36,31,32,32,10,NaN
1,10,31,52,165,225,221,150,143,48,31,...,111,125,143,159,69,45,32,71,10,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99998,10,31,66,107,142,155,142,107,66,31,...,93,66,93,93,66,31,31,31,10,4.0
99999,10,32,41,21,45,89,45,60,67,31,...,97,82,74,80,52,31,32,7,10,2.0


In [ ]:
df.info()   # observe datatypes and any missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Columns: 287 entries, A0T0G0C10 to y
dtypes: float64(1), int64(286)
memory usage: 219.0 MB


In [ ]:
vX = df.query('y!=y').drop('y', axis=1)    # slice a test sample
tXY = df.query('y==y')                     # slice training sample
tX, tY = tXY.drop('y', axis=1), tXY.y.astype(int)      # split into training I/O
vX  # test inputs
tX  # train inputs
print(tY.tolist()[:50]) # train outputs

,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G0C2,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0
0,10,32,41,39,77,122,55,81,58,31,...,27,38,88,80,20,36,31,32,32,10
1,10,31,52,165,225,221,150,143,48,31,...,76,111,125,143,159,69,45,32,71,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9998,10,31,66,107,142,156,143,107,66,31,...,420,296,303,421,302,306,31,31,31,10
9999,10,31,66,107,142,155,142,108,66,31,...,66,429,307,93,93,66,31,31,31,10


,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G0C2,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0
10000,10,7,48,74,119,110,127,86,66,7,...,223,319,233,361,439,368,179,174,199,10
10001,10,31,66,107,142,156,142,108,66,31,...,66,539,302,93,424,434,31,313,31,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99998,10,31,66,107,142,155,142,107,66,31,...,66,93,66,93,93,66,31,31,31,10
99999,10,32,41,21,45,89,45,60,67,31,...,52,97,82,74,80,52,31,32,7,10


[3, 3, 4, 0, 4, 3, 4, 4, 3, 3, 2, 2, 4, 2, 2, 4, 4, 2, 2, 0, 2, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 4, 4, 2, 3, 3, 2, 0, 1, 4, 4, 2, 3, 4, 3, 0, 3, 2, 2, 0]


In [ ]:
tmr = Timer()

⏳ started. You have 60 sec. Good luck!


<hr color=green size=40>

<strong><font color=green size=5>⏳Timed Green Playground (TGP): Your ideas, code, documentation, and timer START HERE!</font></strong>

<font color=green>Students: Keep all your definitions, code, documentation in <b>TGP</b>. Modifying any code outside of TGP incurs penalties.

<font color=green><h3><b>Import Necessary Libraries</b><h3>

In [ ]:
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Nadam
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE
from collections import Counter
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import random

<font color=green><h3><b>Set Random State for Replication</b><h3>

In [ ]:
os.environ['PYTHONHASHSEED'] = str(0)
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)
Init = keras.initializers.RandomNormal(seed=0)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

<font color=green><h3><b>Splitting Dataset for Validation and Over/Under Sampling Training Data</b><h3>

In [ ]:
# Train/Validation Split
tX_train, tX_val, tY_train, tY_val = train_test_split(tX, tY, test_size=0.20, stratify=tY, random_state=0, shuffle=True)

# Class Distribution
class_counts = Counter(tY_train)
total_samples = len(tY_train)

sampling_strategy = {
            3: int(class_counts[3] * 0.9),  # Reduce slightly (~90%)
            4: int(class_counts[4] * 0.9),  # Reduce slightly (~90%)
            2: class_counts[2],  # Keep stable (~100%)
            0: int(class_counts[0] * 3),  # Increase (~180%)
            1: int(class_counts[1] * 3),  # Increase (~180%)
        }

# Oversampling Minority Classes
smote = SMOTE(sampling_strategy={0: sampling_strategy[0], 1: sampling_strategy[1]}, random_state=0)
tX_resampled, tY_resampled = smote.fit_resample(tX_train, tY_train)

# Undersampling Majority Classes
undersample = RandomUnderSampler(sampling_strategy={3: sampling_strategy[3], 4: sampling_strategy[4]}, random_state=0)
tX_train, tY_train = undersample.fit_resample(tX_resampled, tY_resampled)

<font color=green><h3><b>Pre-Processing: Power Transform, Scaling, PCA</b><h3>

In [ ]:
preprocessing_pipeline = Pipeline([
    ("power_transform", PowerTransformer(method="yeo-johnson")),  # Feature Transformation
    ("scaler", StandardScaler()),                                 # Standardization
    ("pca", PCA(n_components=0.99))                               # Dimensionality Reduction
])

tX_train = preprocessing_pipeline.fit_transform(tX_train)
tX_val = preprocessing_pipeline.transform(tX_val)
vX_test = preprocessing_pipeline.transform(vX)

<font color=green><h3><b>Build and Fit Model</b><h3>

In [ ]:
# Define Model Architecture
model = Sequential([
            Flatten(input_shape=[tX_train.shape[1]]),
            Dense(512, activation="relu", kernel_initializer=Init, kernel_regularizer=l2(0.00005)),
            Dense(128, activation="relu", kernel_initializer=Init, kernel_regularizer=l2(0.0005)),
            Dense(256, activation="relu", kernel_initializer=Init, kernel_regularizer=l2(0.0003)),
            Dense(len(tY.unique()), activation='softmax')
        ])

# Define Early Stopping, Optimizer and Learning Scheduler
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=0)
optimizer = Nadam(learning_rate=0.0005)

# Compile Model
model.compile(
      optimizer=optimizer,
      loss='sparse_categorical_crossentropy',
      metrics=['accuracy']
      )

# Train Model
model.fit(
    tX_train, tY_train,
    epochs=18, batch_size=512,
    validation_data=(tX_val, tY_val),
    callbacks=[early_stopping, scheduler],
    verbose=0
)

<font color=green><h3><b>Test on Validation Set</b><h3>

In [ ]:
# Predict and Evaluate
#Y_pred_val = np.argmax(model.predict(tX_val), axis=-1)
#f1 = f1_score(tY_val, Y_pred_val, average='macro')
#print(f"F1 Score: {f1}")

<font color=green><h3><b>Predict on Test Set and Output Data</b><h3>

In [ ]:
pY = pd.DataFrame(np.argmax(model.predict(vX_test), axis=-1), index=vX.index+1, columns=['y'])
ToCSV(pY.round(0).astype(int), 'attempt')
print('Observed  y distribution: ', list((tXY.y.value_counts()/len(tXY)).round(3)))                # distribution of training target level
print('Predicted y distribution: ', list((pY.round(0).astype(int).value_counts()/len(pY)).round(3))) # distribution of test target level

313/313 [==============================] - 1s 2ms/step
Observed  y distribution:  [0.303, 0.302, 0.296, 0.05, 0.048]
Predicted y distribution:  [0.303, 0.303, 0.296, 0.049, 0.049]


<font color=green><h3><b>$\delta$. Idea Documentation</b></h3>
<details>
  <summary>Instructions</summary>
  <div>


1. **Audience**. Your peers who will learn from your Colab and ideas therein.
1. **Importance**. The ML/DL ideas are not entirely random, but are based on prior experience and systematized/organized experiments. We'd like students to share and learn from idea generation to idea experimentation process done in our class using tools learned thus far.
1. **Format**. Keep it concise/precise in consistent font/presentation. Include numbers/IDs to your References, such as [1] or [[Géron22]](https://scholar.google.com/scholar?cluster=498861685923226475), where these are defined in your References section below. This helps link your ideas/experiments to external ideas.
1. **Reproducibility**. Your description should contain reasonable details needed for reproducibility, i.e. describe the state of your modeling pipeline before the change is made, what is changed and how the idea was discovered, and what improvement it resulted in. Thus, peers can try this idea with an expectation of the value it brings. See examples below.
1. **Bonus** points for the exceptional/exemplary/educational documentation (see grading rubric).
****
1. **TODO**: Describe the key idea in your work in the following format (similar to a "micro publication"):
  1. **Title**. Give each idea a descriptive name (i.e. a micro abstract).
    1. Ex(ample). <i>"Thresholding carat feature outliers improves MAE by 3% on public LB"</i>
  1. **Idea Discovery**. What led you to this idea? Was it some [EDA](https://en.wikipedia.org/wiki/Exploratory_data_analysis), familiarity with this dataset or some of the features?
    1. Ex. <i>"We plotted all univariate distributions of variables and discovered that diamond carat had unreasonable (but rare) values below and above [0,10] interval, when plotted carat's histogram in the train and test sets, which contained 10 and 3 such outliers respectively. We decided to use 10 as a reasonable threshold because it is 99th percentile of carat values in the 20K baseline sample. See our histogram plot below [plot here]. "</i>
  1. **Finding's Importance**. Describe why you think the idea was important to proceed with.
    1. Ex. <i>"We use a linear model, the slope of which is sensitive to outliers on the periphery of the feature space domain. The fitted hyperplane slopes in the direction of the extreme training feature values thereby mapping a non-existent relation between carat size and diamond price, which is not expected to repeat in the test set. "</i>
  1. **Experiment Setup**.
  How did you set up experiments to test your idea? What resources were helpful? What metric did you select, why and what values did you observe?
    1. Ex. <i>"To alleviate the impact of the outlying feature values, we need to either remove observations with extreme values, or somehow cap them (to stay within the distribution of the other carat values) or use a model insensitive to outliers (such as robust regression). We learned 3 suitable methods for treating outliers in [ref]: ... [It'd be great to briefly describe each method] We tried each one on a Baseline model, while keeping the competition-required [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) metric. We tested each method locally on the seeded 50/50 split of the 20K training set sampled in baseline Colab."</i>
  1. **Results**. What was the result or metric improvement from implementing the experiment locally and/or on public LB?
    1. Ex. <i>"Baseline MAE was 539.1257546465 in public LB and 530 in local default experiment with 50/50 train-test split. When applied on the same-seed split, Methods 1,2,and 3 showed 1%, 2%, and 5% improvement on the test set. When uploaded to public LB, Method 3 showed a 3% improvement. So, we decided to keep method 3."</i>

</div> </details>
</font>


<font color=green><h4><b>Task 1. Preprocessing Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped in <b>preprocessing pipeline</b>. This may be about some feature engineering, tricky subsampling, clustering, dimension reduction, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: Hybrid Sampling Strategy for Imbalanced Classes
1. **Idea Discovery**: The dataset presented showed a severe class imbalance with three classes accounting for ~30% of the dataset respectively, while the classes 0 and 1 only being ~5% of the dataset respectively. The imbalance led to misprediction of the minority classes when inputted into the deep neural network as is. Readjusting the distribution of the dataset may allow for better learning of the minority classes, without domination of the majority classes.
1. **Finding's Importance**: Class imbalance can negatively impact deep learning models, with a preference to prioritize majority classes. While a high accuracy can be achieved, a low macro F1-score can occur due to improper learning of the minority classes. Several methods to reduce the imbalance of the dataset were explored. SMOTE (Synthetic Minority Oversampling Technique) synthetically generates samples allowing for augmentation of minority classes. Borderline SMOTE specifically generates samples near decision boundaries to differentiate class differences better. Random undersampling allows for reducing the number of majority class samples to prevent model bias, though excessive undersampling risks losing valuable information. Class weighting can be used as a non-sampling technique, adjusting the loss function to penalize misclassification of minority classes without altering the dataset itself [1].
1. **Experiment Setup**: Multiple approaches were implemented and compared to a baseline, where no changes were made to the dataset. The use of only SMOTE and Borderline SMOTE were assessed with application on the minority classes 0 and 1. The use of Random Undersampling was assessed with application on the majority classes 2, 3, and 4. A hybrid approach was developed using both oversampling and undersampling methods combined, with fine-tuned weightages for each class. Beyond resampling, class weighting was tested, where the loss function was adjusted to penalize misclassification of minority classes more heavily, aiming to naturally guide the model toward better learning without modifying the dataset itself. The models trained on these resampled datasets were validated using a three-fold cross-validation strategy, ensuring a fair evaluation of each sampling technique. The impact of each approach was assessed based on the macro F1-score, which provided a more balanced evaluation by weighing each class equally.
1. **Results**: Oversampling with SMOTE significantly increased the dataset size, leading to long training times that exceeded the 60 second limitation. Fully balancing the dataset also resulted in an unnatural class distribution, causing the model to learn unrealistic patterns that did not generalize well. Borderline SMOTE had a similar result. On the other hand, random undersampling removed too many samples from the majority classes, leading to a severe loss of information. Class weighting, while computationally efficient, proved ineffective, as the model still struggled to properly classify the minority classes. The hybrid sampling strategy (combining SMOTE for minority class augmentation with controlled undersampling of the majority classes) yielded the best results. This approach allowed for fine-tuned adjustments to class proportions, maintaining a natural distribution without excessive synthetic data generation or information loss. The optimized hybrid strategy significantly improved the macro F1-score while keeping the dataset at a manageable size for training efficiency.

1. **Title**: Data Transformation, Polynomial Features and Scaling
1. **Idea Discovery**: Initial experiments using the raw feature set resulted in unstable model training, likely due to the presence of highly skewed feature distributions and varying scales across different features. To address these issues, multiple feature transformation techniques were explored. Transformations such as logarithmic, square root, and power-based functions were considered to reduce skewness and enhance separability between classes. Feature scaling was examined to normalize feature distributions and ensure that all features contribute equally to the optimization process. Polynomial feature expansion was also explored as a potential method to capture nonlinear interactions between features.
1. **Finding's Importance**: Deep learning models can be highly sensitive to feature distributions, and improper scaling or skewed distributions, leading to exploding or vanishing gradients, poor convergence rates, and suboptimal learning [1]. Feature transformation methods such as log, square root, and power transformations help adjust feature distributions closer to a normal distribution, improving stability during optimization. Similarly, feature scaling ensures that all features contribute equally to model training, preventing certain features with larger magnitudes from dominating the loss function. Polynomial feature expansion can be particularly useful in capturing higher-order interactions between features, but it also comes with a significant computational cost, especially when applied to a high-dimensional feature space. Identifying the optimal combination of transformations and scaling techniques was crucial for stabilizing training and achieving ideal F1 scores.
1. **Experiment Setup**: Three different scaling techniques were compared: StandardScaler (z-score normalization), MinMaxScaler (scaling to a fixed range), and RobustScaler (scaling based on percentiles to reduce sensitivity to outliers). :og, square root, and power transformations (Yeo-Johnson and Box-Cox) were applied to assess their effect on feature distribution normalization. To test the impact of polynomial feature expansion, second-degree polynomial interactions were generated and evaluated. Each transformation and scaling technique was tested within the preprocessing pipeline, and model performance was assessed using 3 cross-validation with macro F1-score as the primary evaluation metric.
1. **Results**: The best pre-processing framework included PowerTransformer (Yeo-Johnson) and StandardScaler without polynomial expansion. This combination of data transformations resulted in a more normalized feature distribution. Polynomial expansion was found to be ineffective on the already-large feature set, leading to incredibly long runtimes and issues with excessive memory usage. StandardScaler outperformed MinMaxScaler and Robust Scaler. The Yeo-Johnson PowerTransformer outperformed Box-Cox transformation, square root and logarithmic transformations, allowing for better class separation and more normal distribution.

1. **Title**: Dimensionality Reduction
1. **Idea Discovery**: The dataset contained 286 features per bacteria sample. This high-dimensional representation raised concerns about redundancy and computational inefficiency. Principal Component Analysis (PCA) and Linear Discriminant Analysis (LDA) were explored as dimensionality reduction techniques to retain the most informative components while discarding noise.
1. **Finding's Importance**: High-dimensional datasets pose the issue of high computational costs, risk of overfitting due to redundancy and longer training times. Many features can be correlated or redundant, overburdening the learning and predicting process. PCA, an unsupervised technique, identifies the most significant variance-preserving components, while LDA, a supervised technique, maximizes class separability [1].
1. **Experiment Setup**: PCA was tested with variance retention thresholds ranging from 90% to 99.9%, while. LDA was assessed with n_components = 4, as the dataset presents 5 classes. Each method was compared to the baseline model of no dimensionality reduction using 3-cross validation, with macro F1-score as the primary evaluation metric.
1. **Results**: The best model performance was achieved when PCA retained 99% of the variance, balancing dimensionality reduction with the F1 metric. Retaining too few components (<99% variance) resulted in noticeable information loss and reduced classification performance, while keeping too many components (>99%) introduced excessive noise. LDA, despite its theoretical benefits in maximizing class separability, resulted in a 3-4% drop in macro F1-score, suggesting that its assumptions about class distributions did not align well with the dataset.

1. **Title**: Threshold Adjustment for Misclassification
1. **Idea Discovery**: We analyzed the confusion matrix of the hypertuned and identified that classes 0 and 1 were frequently misclassified. These classes made up the minority of the training data, but we wanted to alleviate that difference. We plotted precision-recall curves and revealed that they were most balanced at threshold values 0.25 and 0.3 respectively. Based on this observation, an adjustment was made to the classification threshold to hopefully improve model predictions for these classes.
1. **Finding's Importance**: By adjusting the threshold, the goal was to improve the balance between precision and recall. Because these classes were the most misclassified, we only altered the decision threshold for them.
1. **Experiment Setup**: We first get the probability predicted by the model that an observation is in each class. In altering the decision threshold to 0.25 or 0.3, we say that we should classify the observation as class 0 if the distribution is above the selected threshold.
1. **Results**: After adjusting the threshold, the model’s f1-score dropped from 0.9856489667 to 0.9745186688. This indicates that while threshold tuning altered the class balance, it did not improve overall model performance. We did see an increase in correct classifications for observations in class 0 and class 1, however.

<font color=green><h4><b>Task 2. Modeling Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped with <b>model selection</b> in the format specified above. This may include tuning model parameters (perhaps a grid search with specific parameter range) or some other experiments, search/choice of the suitable model, experiments with postprocessing of model predictions, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: Model Development and Hyperparameter Optimization
1. **Idea Discovery**: The model architecture and hyperparameters play a crucial role in its performance. Given the complexity of the dataset and the need for generalization, we systematically optimized multiple components of the model using grid search. This involved testing different architectures, activation functions, optimizers, batch sizes, regularization strategies, and learning rate schedules. The goal was to create a model that could maximize macro F1-score while maintaining stability, efficiency, and reproducibility, paying close attention to not only the F1 score but also the standard deviation across cross validation folds.
1. **Finding's Importance**: Each component of the model affects learning in a unique and impactful way. The optimizer determines how weights are updated during training, influencing convergence speed and stability. The learning rate and its scheduling impact whether the model learns effectively or gets stuck in local minima. The activation function defines how neurons process information, affecting the model’s ability to capture nonlinear patterns. Regularization strategies help prevent overfitting by constraining weight updates. The number of layers and neurons dictates the model’s capacity to learn complex patterns, while the batch size affects computational efficiency and gradient updates. Different weight initializers can influence early training dynamics and overall stability. Fine-tuning each of these elements ensures that the model generalizes well across unseen data while avoiding inefficiencies or instability. The goal was to not only prioritize high mean F1 scores across validation folds, but also assess standard deviation. Choosing parameters that also resulted in low standard deviation suggested less overfitting and better generalizability.
1. **Experiment Setup**: A 5-fold cross validation setup was used to evaluate the different configurations of parameters, using macro F1 score as the metric of comparison.
  * **Optimizers:** SGD, Adam, AdamW, RMSprop, Nadam
  * **Learning Rate:** 0.001, 0.0005, 0.0003, 0.0001, 0.00005
  * **Learning Rate Schedulers:** ReduceLROnPlateau, Fixed LR, Exponential Decay, Cosine Annealing
  * **Activation Functions:** ReLU, Swish, Tanh, ELU, GELU, LeakyRelu
  * **Regularization:** L2 regularization at different values per layer (0.00005, 0.0001, 0.0005, 0.001)
  * **Network Architecture:** Number of layers (2, 3, 4) and neuron counts per layer (64, 128, 256, 512)
  * **Batch Size:** 128, 256, 512, 1024
  * **Weight Initializer:** Random Normal Initialization, He Normal Initialization (hypothesized to work better with ReLu [1])
  * **Feature Engineering:** Testing higher-order interactions, polynomial features (removed due to excessive computational cost)
  * **Outlier Handling:** IQR filtering, Box-Cox transformations
  * **Loss Functions:** Sparse Categorical Crossentropy, Focal Loss (removed due to performance issues), Cosine Similarity (removed due to instability)

1. **Results**: The best configuration was seen to be:
  * **Optimizer:** Nadam (faster and more accurate than Adam and AdamW)
  * **Learning Rate:** 0.0005
  * **Learning Rate Scheduler:** ReduceLROnPlateau (dynamic adjustment led to better stability)
  * **Activation Function:** ReLU (performed better than other nonlinear activations)
  * **Regularization:** L2 regularization (0.00005 for first layer, 0.0005 for second layer, 0.0003 for third layer)
  * **Architecture:** 3-layer network with 512, 128, and 256 neurons per layer
  * **Batch Size:** 512 (optimal balance of speed and stability)
  * **Weight Initializer:** Random Normal Initialization (seeded for reproducibility)
  * **Feature Engineering:** Power Transformer with Standard Scaling for normalization
  * **Outlier Handling:** No removal due to excessive data loss
  * **Loss Function:** Sparse Categorical Crossentropy (macro F1 performed best with this)
  
This combination of parameters resulted in both high mean F1 scores and low standard deviation, suggesting a highly accurate and robust approach.


<font color=green><h3><b>$\epsilon$. References</b></h3>
<details>
  <summary>Instructions</summary>
  <div>

1. Cite your sources to help your peers learn from these (and to avoid plagiarism).
1. HOML textbook should be cited, since we used it in this week's learning.
1. Use Google Scholar to draw [APA](https://en.wikipedia.org/wiki/American_Psychological_Association) citation format for books and publications.
1. Cite [StackOverflow](https://stackoverflow.com/), YouTube videos, package docs, open-access textbooks/publicaitons and other meaningful internet resources that you used.
1. We may reward exceptional and meaningful citations (not just a list of [SKL](https://scikit-learn.org/stable/)/[TF](https://www.tensorflow.org/) manual pages and a list of articles.) For example, if you used an idea from a publication, indicate it in TGP with a number that corresponds to its reference in References.

</div> </details>
</font>

1. Geron, A. (2019). Hands-On Machine Learning with Scikit-Learn, Keras & Tensorflow. 2nd Ed., Sebastopol, CA: O’Reilly, 2019.


<font size=5>⌛</font> <strong><font color=green size=5>Do not exceed competition's runtime limit! Do not write code outside TGP</font></strong>
<hr color=green size=40>

In [ ]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 55 sec


<details>
  <summary><font size=5><b>💡Starter Ideas</b></font></summary>
  <div>
  
**Model**
1. Tune model hyperparameters, batch size, optimizer, NN layers

**Features**
1. Try to linear and non-linear feature normalization: shift/scale, log, divide features by features (investigate scatterplot matrix)
1. Try higher order feature interactions and polynomial features on a small subsample. Then identify key features or select key principal components. The final model can be trained on a larger or even full training sample. You can use [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) to reduce the feature set

**Training observations**
1. Try clustering similar observations into fewer representative observations.
1. You may also try dimension reduction methods (eg. PCA) on the transposed data matrix (if it has scaled numeric features).
1. Look for and deal with outliers or influential points in the training set
1. Deal with **imbalanced sample**: oversample smaller class, or undersample larger class, or provide observation weights or provide class weights, or seek a suitable loss function
1. Investigate distributions of features. Any missing values? Any zero values?

**Predictions**
1. Evaluate predictions and focus on poorly predicted "groups":
  1. Strongest misclassifications. E.g. the model is very confident about the wrong label
  1. Evaluate predictions near decision boundaries.

**EDA and Domain Expertise**
1. Do a thorough EDA: look for feature augmentations that result in linear decision boundaries between pairs of classes.
1. Learn about the domain. Read [Analysis of Identification Method for Bacterial Species and Antibiotic Resistance Genes Using Optical Data From DNA Oligomers](https://www.frontiersin.org/articles/10.3389/fmicb.2020.00257/full) and cited/citing references.

</div> </details>